# Preprocessing comparison (a) · Training on the background-removed copy
Repeats light-only training on the background-removed Kaggle copy of Fitzpatrick17k (Nazmus Resan). Used only for the preprocessing comparison; the main results use the original images (notebooks 01–05).

**Kaggle inputs:** `fitzpatrick17k` (Nazmus Resan). GPU (T4), internet on.

## 1. Write the scripts to disk

In [ ]:
%%writefile skin_tone_gap.py"""Skin-tone generalisation gap in dermatology classifiers.Train on light skin (FST I-III), evaluate on light AND dark (FST IV-VI).USAGE-----Smoke test first (2 min, proves the pipeline runs):    SMOKE=1 python skin_tone_gap.pyFull run, one seed:    SEED=0 python skin_tone_gap.pyMitigation run (oversamples dark skin in training):    SEED=0 MITIGATE=1 python skin_tone_gap.pyIn a Kaggle notebook cell:    import os; os.environ["SMOKE"] = "1"    %run skin_tone_gap.pyOUTPUTS (to OUT_DIR)--------------------    results_seed{N}.json      per-group metrics    probs_seed{N}.csv         raw per-image probabilities  <-- keep these!    model_seed{N}.pt          checkpoint for Grad-CAM laterThe probs CSV is the important artefact: it lets you compute NEW metricstomorrow (ECE, bootstrapped CIs, subgroup splits) WITHOUT retraining."""import osimport globimport jsonimport randomimport warningsimport numpy as npimport pandas as pdimport torchimport torch.nn as nnfrom PIL import Imagefrom sklearn.metrics import roc_auc_score, confusion_matrixfrom torch.utils.data import Dataset, DataLoader, WeightedRandomSamplerfrom torchvision import transforms, modelswarnings.filterwarnings("ignore")# ----------------------------------------------------------------------------# CONFIG# ----------------------------------------------------------------------------SEED      = int(os.environ.get("SEED", 0))SMOKE     = os.environ.get("SMOKE", "0") == "1"MITIGATE  = os.environ.get("MITIGATE", "0") == "1"EVAL_ONLY = os.environ.get("EVAL_ONLY", "0") == "1"   # load checkpoint, skip trainingRESUME    = os.environ.get("RESUME", "0") == "1"      # continue from checkpoint# MIT_TARGET: which FST levels the mitigation oversamples.#   "dark" = all of FST 4-6 (the naive version)#   "6"    = FST 6 only -- use this if the cliff is at FST 6, not across 4-6MIT_TARGET = os.environ.get("MIT_TARGET", "dark")# HOLDOUT_FST: train on light + these FST levels, test on the rest.#   e.g. HOLDOUT_FST="4,5" trains on light+FST4+FST5 and tests on FST6.#   Isolates "never saw dark skin" from "FST 6 is intrinsically harder".HOLDOUT_FST = os.environ.get("HOLDOUT_FST", "")IMG_SIZE   = 224BATCH_SIZE = 32EPOCHS     = 2 if SMOKE else int(os.environ.get("EPOCHS", 10))LR         = 3e-4N_SMOKE    = 200          # images used in smoke modeDATA_ROOT = os.environ.get("DATA_ROOT", "/kaggle/input")OUT_DIR   = os.environ.get("OUT_DIR", "/kaggle/working")os.makedirs(OUT_DIR, exist_ok=True)DEVICE = ("cuda" if torch.cuda.is_available()          else "mps" if torch.backends.mps.is_available()          else "cpu")tag = (f"seed{SEED}"       + (f"_mitigate{MIT_TARGET}" if MITIGATE else "")       + (f"_hold{HOLDOUT_FST.replace(',', '')}" if HOLDOUT_FST else "")       + ("_smoke" if SMOKE else ""))def set_seed(s):    random.seed(s); np.random.seed(s); torch.manual_seed(s)    torch.cuda.manual_seed_all(s)# ----------------------------------------------------------------------------# 1. FIND THE DATA#    Kaggle mirrors differ. We sniff for the CSV and the image folder.# ----------------------------------------------------------------------------def find_csv():    cands = glob.glob(f"{DATA_ROOT}/**/*.csv", recursive=True)    if not cands:        raise FileNotFoundError(f"No CSV under {DATA_ROOT}. Is the dataset attached?")    # prefer one that looks like the fitzpatrick index    for c in cands:        if "fitz" in os.path.basename(c).lower():            return c    return max(cands, key=os.path.getsize)def build_image_index():    """Map stem -> full path for every image under DATA_ROOT."""    idx = {}    for ext in ("jpg", "jpeg", "png", "JPG", "JPEG", "PNG"):        for p in glob.glob(f"{DATA_ROOT}/**/*.{ext}", recursive=True):            idx[os.path.splitext(os.path.basename(p))[0]] = p    if not idx:        raise FileNotFoundError(f"No images found under {DATA_ROOT}")    return idxdef pick_col(df, options, what):    low = {c.lower(): c for c in df.columns}    for o in options:        if o in low:            return low[o]    raise KeyError(        f"Could not find a {what} column. Looked for {options}.\n"        f"Columns present: {list(df.columns)}\n"        f"--> Edit pick_col() with the right name."    )print(f"[setup] device={DEVICE}  seed={SEED}  smoke={SMOKE}  mitigate={MITIGATE}")csv_path = find_csv()df = pd.read_csv(csv_path)print(f"[data] csv: {csv_path}  rows={len(df)}")print(f"[data] columns: {list(df.columns)}")FST_COL   = pick_col(df, ["fitzpatrick_scale", "fitzpatrick", "fitzpatrick_centaur",                          "fst", "skin_type"], "Fitzpatrick")LABEL_COL = pick_col(df, ["three_partition_label", "label", "diagnosis",                          "nine_partition_label", "target"], "label")ID_COL    = pick_col(df, ["md5hash", "image_id", "filename", "image", "id",                          "url_alphanum"], "image id")print(f"[data] using  fst={FST_COL}  label={LABEL_COL}  id={ID_COL}")# ----------------------------------------------------------------------------# 2. BUILD LABELS AND GROUPS# ----------------------------------------------------------------------------df[FST_COL] = pd.to_numeric(df[FST_COL], errors="coerce")df = df[df[FST_COL].between(1, 6)].copy()          # drop -1 / unknowndf["group"] = np.where(df[FST_COL] <= 3, "light", "dark")# Binary target: malignant vs not. Adjust the keyword if your label space differs.lab = df[LABEL_COL].astype(str).str.lower()df["y"] = lab.str.contains("malignant").astype(int)if df["y"].nunique() < 2:    raise ValueError(        f"Binary target collapsed. Unique {LABEL_COL} values: "        f"{df[LABEL_COL].unique()[:20]}\n--> Fix the 'malignant' rule above."    )img_index = build_image_index()df["path"] = df[ID_COL].astype(str).map(img_index)missing = df["path"].isna().sum()df = df.dropna(subset=["path"]).reset_index(drop=True)print(f"[data] resolved {len(df)} images ({missing} unresolved / URL-rotted)")print("\n[data] group x class counts")print(pd.crosstab(df["group"], df["y"]))# ----------------------------------------------------------------------------# 3. SPLIT  -- train on LIGHT only; test on both groups# ----------------------------------------------------------------------------set_seed(SEED)light = df[df.group == "light"].sample(frac=1, random_state=SEED).reset_index(drop=True)dark  = df[df.group == "dark"].reset_index(drop=True)n_test = int(0.20 * len(light))test_light  = light.iloc[:n_test]train_light = light.iloc[n_test:]if HOLDOUT_FST:    # train on light + the named FST levels; test on the FST levels NOT named    keep = [int(v) for v in HOLDOUT_FST.split(",")]    train_extra = dark[dark[FST_COL].isin(keep)]    test_dark   = dark[~dark[FST_COL].isin(keep)]    train = pd.concat([train_light, train_extra], ignore_index=True)    print(f"[holdout] training also on FST {keep}; testing on FST "          f"{sorted(test_dark[FST_COL].unique().astype(int))}")elif MITIGATE:    # hold out half the targeted data for testing, put the rest in training    pool = dark if MIT_TARGET == "dark" else dark[dark[FST_COL] == int(MIT_TARGET)]    rest = dark.drop(pool.index)    pool = pool.sample(frac=1, random_state=SEED)    cut = len(pool) // 2    test_dark   = pd.concat([pool.iloc[:cut], rest], ignore_index=True)    train_extra = pool.iloc[cut:]    train = pd.concat([train_light, train_extra], ignore_index=True)    print(f"[mitigate] oversampling target='{MIT_TARGET}', "          f"{len(train_extra)} images added to training")else:    test_dark = dark    train = train_lightif SMOKE:    train     = train.sample(min(N_SMOKE, len(train)), random_state=SEED)    test_light = test_light.sample(min(100, len(test_light)), random_state=SEED)    test_dark  = test_dark.sample(min(100, len(test_dark)),  random_state=SEED)print(f"\n[split] train={len(train)}  test_light={len(test_light)}  test_dark={len(test_dark)}")print(f"[split] test_dark positives={int(test_dark.y.sum())}")if len(test_dark) < 200 and not SMOKE:    print("[WARN] dark test set is small -- expect wide CIs. "          "Consider bootstrapping rather than relying on seed variance alone.")# ----------------------------------------------------------------------------# 4. DATA PIPELINE# ----------------------------------------------------------------------------NORM = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])train_tf = transforms.Compose([    transforms.Resize((IMG_SIZE, IMG_SIZE)),    transforms.RandomHorizontalFlip(),    transforms.RandomVerticalFlip(),    transforms.RandomRotation(20),    transforms.ToTensor(), NORM,])# NOTE: deliberately NO colour jitter -- it would confound the skin-tone signal.eval_tf = transforms.Compose([    transforms.Resize((IMG_SIZE, IMG_SIZE)),    transforms.ToTensor(), NORM,])class SkinDS(Dataset):    def __init__(self, frame, tf):        self.df, self.tf = frame.reset_index(drop=True), tf    def __len__(self):        return len(self.df)    def __getitem__(self, i):        r = self.df.iloc[i]        img = Image.open(r["path"]).convert("RGB")        return self.tf(img), int(r["y"]), iNW = int(os.environ.get("NUM_WORKERS", 4 if DEVICE == "cuda" else 0))def loader(frame, tf, shuffle=False, sampler=None):    return DataLoader(SkinDS(frame, tf), batch_size=BATCH_SIZE,                      shuffle=shuffle and sampler is None,                      sampler=sampler, num_workers=NW, pin_memory=(DEVICE == "cuda"))sampler = Noneif MITIGATE:    # oversample the targeted images so the model sees them near parity    if MIT_TARGET == "dark":        is_target = (train["group"].values == "dark")    else:        is_target = (train[FST_COL].values == int(MIT_TARGET))    n_t = max(is_target.sum(), 1)    n_o = max((~is_target).sum(), 1)    w = np.where(is_target, len(train) / n_t, len(train) / n_o)    sampler = WeightedRandomSampler(torch.DoubleTensor(w), len(train), replacement=True)    print(f"[mitigate] sampler weights: {n_t} target vs {n_o} other")train_dl = loader(train, train_tf, shuffle=True, sampler=sampler)light_dl = loader(test_light, eval_tf)dark_dl  = loader(test_dark,  eval_tf)# ----------------------------------------------------------------------------# 5. MODEL# ----------------------------------------------------------------------------model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)model.fc = nn.Linear(model.fc.in_features, 2)model = model.to(DEVICE)# class-weighted loss: malignant is the minority and the one that matterscounts = train["y"].value_counts()cw = torch.tensor([len(train) / (2 * counts.get(0, 1)),                   len(train) / (2 * counts.get(1, 1))],                  dtype=torch.float32, device=DEVICE)crit = nn.CrossEntropyLoss(weight=cw)opt  = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))# ----------------------------------------------------------------------------# 6. TRAIN# ----------------------------------------------------------------------------CKPT = f"{OUT_DIR}/model_{tag}.pt"if EVAL_ONLY or RESUME:    if not os.path.exists(CKPT):        raise FileNotFoundError(            f"No checkpoint at {CKPT}. The session was wiped -- retrain from scratch."        )    model.load_state_dict(torch.load(CKPT, map_location=DEVICE))    print(f"[ckpt] loaded {CKPT}")if EVAL_ONLY:    print("[eval] skipping training, going straight to evaluation")else:    for ep in range(EPOCHS):        model.train()        tot = 0.0        for x, y, _ in train_dl:            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)            opt.zero_grad(set_to_none=True)            with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):                loss = crit(model(x), y)            scaler.scale(loss).backward()            scaler.step(opt); scaler.update()            tot += loss.item() * x.size(0)        sched.step()        print(f"[train] epoch {ep+1}/{EPOCHS}  loss={tot/len(train_dl.dataset):.4f}",              flush=True)   # flush: Kaggle buffers stdout and looks frozen otherwise        # checkpoint EVERY epoch -- Kaggle sessions die without warning        torch.save(model.state_dict(), CKPT)# ----------------------------------------------------------------------------# 7. EVALUATE# ----------------------------------------------------------------------------@torch.no_grad()def predict(dl):    model.eval()    P, Y, I = [], [], []    for x, y, i in dl:        p = torch.softmax(model(x.to(DEVICE)), 1)[:, 1]        P.append(p.cpu().numpy()); Y.append(y.numpy()); I.append(i.numpy())    return np.concatenate(P), np.concatenate(Y), np.concatenate(I)def ece(probs, y, bins=10):    """Expected calibration error -- keep this even if the accuracy gap is small."""    conf = np.where(probs >= 0.5, probs, 1 - probs)    correct = ((probs >= 0.5).astype(int) == y).astype(float)    edges, e = np.linspace(0, 1, bins + 1), 0.0    for lo, hi in zip(edges[:-1], edges[1:]):        m = (conf > lo) & (conf <= hi)        if m.sum():            e += m.mean() * abs(correct[m].mean() - conf[m].mean())    return float(e)def metrics(probs, y):    pred = (probs >= 0.5).astype(int)    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()    return {        "n": int(len(y)),        "n_pos": int(y.sum()),        "sensitivity": float(tp / (tp + fn)) if (tp + fn) else None,        "specificity": float(tn / (tn + fp)) if (tn + fp) else None,        "auc": float(roc_auc_score(y, probs)) if len(np.unique(y)) > 1 else None,        "ece": ece(probs, y),        "accuracy": float((pred == y).mean()),    }res, rows = {}, []for name, dl, frame in [("light", light_dl, test_light), ("dark", dark_dl, test_dark)]:    p, y, i = predict(dl)    res[name] = metrics(p, y)    sub = frame.reset_index(drop=True).iloc[i]    rows.append(pd.DataFrame({        "group": name, "prob": p, "y": y,        "path": sub["path"].values, "fst": sub[FST_COL].values,        # carry the diagnosis through so the disease-mix confound can be        # checked later WITHOUT retraining        "condition": sub[LABEL_COL].values,    }))if res["light"]["sensitivity"] is not None and res["dark"]["sensitivity"] is not None:    res["gap_sensitivity"] = res["light"]["sensitivity"] - res["dark"]["sensitivity"]if res["light"]["auc"] is not None and res["dark"]["auc"] is not None:    res["gap_auc"] = res["light"]["auc"] - res["dark"]["auc"]res["gap_ece"] = res["dark"]["ece"] - res["light"]["ece"]res["config"] = {"seed": SEED, "smoke": SMOKE, "mitigate": MITIGATE,                 "epochs": EPOCHS, "n_train": len(train)}pd.concat(rows).to_csv(f"{OUT_DIR}/probs_{tag}.csv", index=False)with open(f"{OUT_DIR}/results_{tag}.json", "w") as f:    json.dump(res, f, indent=2)print("\n" + "=" * 60)print(json.dumps(res, indent=2))print("=" * 60)print(f"saved -> {OUT_DIR}/results_{tag}.json  /  probs_{tag}.csv  /  model_{tag}.pt")

In [ ]:
%%writefile analyze_gap.py"""Analyse the skin-tone gap from saved probabilities. No GPU, no retraining.    python analyze_gap.pyTests the key hypothesis: is the sensitivity gap a THRESHOLD artefact(same ranking ability, different operating point) rather than arepresentational failure? AUC ~equal + sensitivity down + specificity upis the signature of exactly that."""import glob, jsonimport numpy as npimport pandas as pdfrom sklearn.metrics import roc_auc_score, roc_curveOUT = "/kaggle/working"files = sorted(glob.glob(f"{OUT}/probs_seed*.csv"))files = [f for f in files if "smoke" not in f and "mitigate" not in f]print(f"found {len(files)} seed(s): {[f.split('/')[-1] for f in files]}\n")def sens_spec(p, y, thr):    pred = (p >= thr).astype(int)    tp = ((pred == 1) & (y == 1)).sum(); fn = ((pred == 0) & (y == 1)).sum()    tn = ((pred == 0) & (y == 0)).sum(); fp = ((pred == 1) & (y == 0)).sum()    return (tp / (tp + fn) if tp + fn else np.nan,            tn / (tn + fp) if tn + fp else np.nan)def thr_at_spec(p, y, target):    """Threshold on this group that achieves a target specificity."""    fpr, tpr, thr = roc_curve(y, p)    i = np.argmin(np.abs((1 - fpr) - target))    return thr[i]def boot_gap(pl, yl, pd_, yd, thr=0.5, n=2000, seed=0):    """Bootstrap CI for the sensitivity gap (light - dark)."""    rng = np.random.default_rng(seed)    gaps = []    for _ in range(n):        il = rng.integers(0, len(yl), len(yl))        id_ = rng.integers(0, len(yd), len(yd))        sl, _ = sens_spec(pl[il], yl[il], thr)        sd, _ = sens_spec(pd_[id_], yd[id_], thr)        if not (np.isnan(sl) or np.isnan(sd)):            gaps.append(sl - sd)    g = np.array(gaps)    return g.mean(), np.percentile(g, 2.5), np.percentile(g, 97.5)rows = []for f in files:    seed = f.split("seed")[-1].split(".")[0]    d = pd.read_csv(f)    L = d[d.group == "light"]; D = d[d.group == "dark"]    pl, yl = L["prob"].values, L["y"].values    pdk, ydk = D["prob"].values, D["y"].values    # --- at the fixed 0.5 threshold ---    sl, spl = sens_spec(pl, yl, 0.5)    sd, spd = sens_spec(pdk, ydk, 0.5)    # --- matched operating point: force dark to light's specificity ---    t_match = thr_at_spec(pdk, ydk, spl)    sd_m, spd_m = sens_spec(pdk, ydk, t_match)    rows.append(dict(        seed=seed,        auc_light=roc_auc_score(yl, pl), auc_dark=roc_auc_score(ydk, pdk),        sens_light=sl, sens_dark=sd, spec_light=spl, spec_dark=spd,        gap_raw=sl - sd,        thr_matched=t_match, sens_dark_matched=sd_m, spec_dark_matched=spd_m,        gap_matched=sl - sd_m,    ))r = pd.DataFrame(rows)pd.set_option("display.width", 200, "display.precision", 4)print(r.to_string(index=False), "\n")print("ACROSS SEEDS (mean +/- std)")for c in ["auc_light", "auc_dark", "sens_light", "sens_dark",          "gap_raw", "gap_matched"]:    print(f"  {c:20s} {r[c].mean():+.4f} +/- {r[c].std():.4f}")print("\nINTERPRETATION")auc_gap = r.auc_light.mean() - r.auc_dark.mean()print(f"  AUC gap            {auc_gap:+.4f}  "      f"({'negligible -> ranking ability is equal' if abs(auc_gap) < 0.02 else 'real ranking difference'})")print(f"  gap at thr=0.5     {r.gap_raw.mean():+.4f}")print(f"  gap spec-matched   {r.gap_matched.mean():+.4f}")shrink = 1 - abs(r.gap_matched.mean()) / max(abs(r.gap_raw.mean()), 1e-9)print(f"  -> matching the operating point removes {shrink*100:.0f}% of the gap")if shrink > 0.6 and abs(auc_gap) < 0.02:    print("  => THRESHOLD ARTEFACT, not a representational failure.")    print("     This is your paper's claim. Per-group thresholds are the remedy.")else:    print("  => gap survives threshold matching: a genuine representational gap.")# --- bootstrap CI on seed 0 (is the raw gap even real?) ---d = pd.read_csv(files[0])L = d[d.group == "light"]; D = d[d.group == "dark"]m, lo, hi = boot_gap(L["prob"].values, L["y"].values,                     D["prob"].values, D["y"].values)print(f"\nBOOTSTRAP (seed {files[0].split('seed')[-1].split('.')[0]}, 2000 resamples)")print(f"  sensitivity gap {m:+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]")print("  " + ("CI excludes zero -> gap is statistically detectable."               if lo > 0 or hi < 0 else               "CI INCLUDES ZERO -> cannot claim a significant gap at this n."))# --- per-FST-level breakdown: AUC is the decisive column ---# A threshold artefact shows: AUC flat, sens down, spec UP.# A real failure shows:       AUC down, sens down, spec down.if "fst" in d.columns:    print("\nPER-FITZPATRICK-LEVEL (seed 0, thr=0.5)")    print(f"  {'FST':>4} {'n':>6} {'pos':>5} {'sens':>7} {'spec':>7} "          f"{'AUC':>7} {'sens_se':>8} {'verdict':>22}")    ref_auc = None    fst_rows = []    for lvl in sorted(d["fst"].dropna().unique()):        s = d[d.fst == lvl]        y, p = s["y"].values, s["prob"].values        if y.sum() < 10 or len(np.unique(y)) < 2:            continue        se_, sp_ = sens_spec(p, y, 0.5)        auc_ = roc_auc_score(y, p)        stderr = np.sqrt(se_ * (1 - se_) / y.sum())   # SE of sensitivity        if ref_auc is None:            ref_auc = auc_        if auc_ < ref_auc - 0.05 and sp_ < 0.90:            verdict = "genuine degradation"        elif se_ < 0.6 and sp_ > 0.92:            verdict = "operating-point shift"        else:            verdict = "-"        print(f"  {int(lvl):>4} {len(s):>6} {int(y.sum()):>5} {se_:>7.3f} "              f"{sp_:>7.3f} {auc_:>7.3f} {stderr:>8.3f} {verdict:>22}")        fst_rows.append((int(lvl), se_, stderr, auc_))    # pairwise test: worst level vs best level    if len(fst_rows) >= 2:        best = max(fst_rows, key=lambda r: r[1])        worst = min(fst_rows, key=lambda r: r[1])        gap = best[1] - worst[1]        gse = np.sqrt(best[2] ** 2 + worst[2] ** 2)        print(f"\n  FST {best[0]} vs FST {worst[0]}: "              f"sensitivity gap {gap:+.3f} +/- {gse:.3f}  "              f"({gap/gse:.1f} SE)")        print("  " + ("SIGNIFICANT -- this, not the light/dark split, is the finding."                      if abs(gap) > 2 * gse else                      "not significant at this n."))        print(f"  AUC at FST {best[0]} = {best[3]:.3f}, at FST {worst[0]} = {worst[3]:.3f}")        print("  -> AUC also down  => genuine representational failure at that level."              if worst[3] < best[3] - 0.05 else              "  -> AUC holds up  => still an operating-point effect.")    print("\n  NOTE: binarising FST 1-3 vs 4-6 averages FST 4/5 (fine) together")    print("        with FST 6 (poor), which is how the aggregate gap loses significance.")

## 2. Main experiment — 5 seedsTrains only on light skin. The entire dark set is held out for testing.~30 min per seed.

In [ ]:
import osos.environ.update(SMOKE="0", EVAL_ONLY="0", MITIGATE="0",                  HOLDOUT_FST="", EPOCHS="10")for s in range(5):    os.environ["SEED"] = str(s)    print(f"\n{'='*70}\nSEED {s}\n{'='*70}", flush=True)    %run skin_tone_gap.py

## 3. Analysis — the verdictThe decisive numbers:- **`gap_matched` ≈ 0 with flat AUC** → the aggregate gap is an operating-point  artefact, not a representational failure.- **FST 6 row** → if sensitivity collapses there while FST 4/5 look fine, the  standard light/dark binarisation is masking a level-specific failure.- **AUC column in the per-FST table** → decides whether FST 6 is a genuine  degradation or just a more extreme threshold shift.

In [ ]:
%run analyze_gap.py

## 4. Mitigation — does oversampling FST 6 help?Half of FST 6 moves into training, oversampled to parity. The other half staysin the test set. 3 seeds is enough for a secondary experiment.Prediction: if FST 6 is a *threshold* effect this barely moves; if it is a*representation* effect it should improve.

In [ ]:
import osos.environ.update(SMOKE="0", EVAL_ONLY="0", EPOCHS="10",                  MITIGATE="1", MIT_TARGET="6", HOLDOUT_FST="")for s in range(3):    os.environ["SEED"] = str(s)    print(f"\n{'='*70}\nMITIGATE FST6 — SEED {s}\n{'='*70}", flush=True)    %run skin_tone_gap.py

## 5. Holdout — is FST 6 unseen, or intrinsically harder?Trains on light **plus FST 4 and 5**, tests on FST 6 only. This separates"the model never saw dark skin" from "FST 6 is hard even with neighbouringtones in training". Nobody frames it this way and it is one extra run.

In [ ]:
import osos.environ.update(SMOKE="0", EVAL_ONLY="0", EPOCHS="10",                  MITIGATE="0", HOLDOUT_FST="4,5")for s in range(3):    os.environ["SEED"] = str(s)    print(f"\n{'='*70}\nHOLDOUT FST6 — SEED {s}\n{'='*70}", flush=True)    %run skin_tone_gap.py

## 6. Confound checksThe two alternative explanations a reviewer will reach for:1. **Disease mix** — is FST 6 loaded with conditions the model handles badly   for reasons unrelated to skin tone?2. **Image quality** — are FST 6 images systematically smaller or blurrier?Both run on CPU from the saved probabilities.

In [ ]:
import pandas as pd, numpy as npfrom PIL import Imaged = pd.read_csv("/kaggle/working/probs_seed0.csv")print("CONDITION MIX BY FST LEVEL (row-normalised)")mix = pd.crosstab(d.fst, d.condition, normalize="index")print((mix * 100).round(1).to_string(), "\n")print("MALIGNANT PREVALENCE BY FST LEVEL")prev = d.groupby("fst")["y"].agg(["mean", "sum", "count"])prev.columns = ["prevalence", "n_pos", "n"]print(prev.round(3).to_string())print("-> if prevalence differs sharply at FST 6, that is a confound to report\n")print("IMAGE DIMENSIONS BY FST LEVEL (sample of 150 per level)")rows = []for lvl, g in d.groupby("fst"):    s = g.sample(min(150, len(g)), random_state=0)    dims = []    for p in s["path"]:        try:            with Image.open(p) as im:                dims.append(im.size)        except Exception:            pass    if dims:        w, h = zip(*dims)        rows.append(dict(fst=int(lvl), n=len(dims),                         mean_w=np.mean(w), mean_h=np.mean(h),                         min_dim=np.mean([min(a, b) for a, b in dims])))print(pd.DataFrame(rows).round(1).to_string(index=False))print("-> systematically smaller FST 6 images would be an alternative explanation")

## 7. FiguresThree figures. Figure 1 (per-FST sensitivity) is the paper's money shot.

In [ ]:
import glob, numpy as np, pandas as pdimport matplotlib.pyplot as pltfrom sklearn.metrics import roc_auc_score, roc_curveplt.rcParams.update({"figure.dpi": 150, "font.size": 9,                     "axes.spines.top": False, "axes.spines.right": False})def sens_spec(p, y, thr=0.5):    pr = (p >= thr).astype(int)    tp = ((pr==1)&(y==1)).sum(); fn = ((pr==0)&(y==1)).sum()    tn = ((pr==0)&(y==0)).sum(); fp = ((pr==1)&(y==0)).sum()    return (tp/(tp+fn) if tp+fn else np.nan, tn/(tn+fp) if tn+fp else np.nan)main = [f for f in sorted(glob.glob("/kaggle/working/probs_seed*.csv"))        if "mitigate" not in f and "hold" not in f and "smoke" not in f]print("main runs:", [f.split('/')[-1] for f in main])# ---- Figure 1: per-FST sensitivity across seeds -----------------------------per = {}for f in main:    d = pd.read_csv(f)    for lvl, g in d.groupby("fst"):        if g.y.sum() >= 10:            se, _ = sens_spec(g["prob"].values, g["y"].values)            per.setdefault(int(lvl), []).append(se)lv = sorted(per); mu = [np.mean(per[l]) for l in lv]; sd = [np.std(per[l]) for l in lv]fig, ax = plt.subplots(figsize=(4.2, 2.8))bars = ax.bar([f"FST {l}" for l in lv], mu, yerr=sd, capsize=3,              color=["#4C72B0"]*len(lv), edgecolor="black", linewidth=.5)if 6 in lv:    bars[lv.index(6)].set_color("#C44E52")ax.set_ylabel("Sensitivity"); ax.set_ylim(0, 1)ax.axhline(np.mean(mu), ls="--", lw=.8, c="gray", label="mean across levels")ax.legend(frameon=False, fontsize=7)ax.set_title(f"Sensitivity by Fitzpatrick level (mean ± SD, {len(main)} seeds)", fontsize=9)plt.tight_layout(); plt.savefig("/kaggle/working/fig1_per_fst.png", bbox_inches="tight")plt.show()# ---- Figure 2: threshold matching ------------------------------------------raw, matched = [], []for f in main:    d = pd.read_csv(f)    L, D = d[d.group=="light"], d[d.group=="dark"]    sl, spl = sens_spec(L["prob"].values, L["y"].values)    sd_, _   = sens_spec(D["prob"].values, D["y"].values)    fpr, tpr, thr = roc_curve(D["y"], D["prob"])    t = thr[np.argmin(np.abs((1-fpr) - spl))]    sdm, _ = sens_spec(D["prob"].values, D["y"].values, t)    raw.append(sl - sd_); matched.append(sl - sdm)fig, ax = plt.subplots(figsize=(3.4, 2.8))ax.bar(["threshold\n= 0.5", "specificity\nmatched"],       [np.mean(raw), np.mean(matched)],       yerr=[np.std(raw), np.std(matched)], capsize=4,       color=["#C44E52", "#55A868"], edgecolor="black", linewidth=.5)ax.axhline(0, c="black", lw=.8)ax.set_ylabel("Sensitivity gap (light − dark)")ax.set_title("The aggregate gap is an\noperating-point effect", fontsize=9)plt.tight_layout(); plt.savefig("/kaggle/working/fig2_threshold.png", bbox_inches="tight")plt.show()# ---- Figure 3: mitigation / holdout comparison ------------------------------def fst6_sens(pattern):    out = []    for f in sorted(glob.glob(pattern)):        d = pd.read_csv(f); g = d[d.fst == 6]        if len(g) and g.y.sum() >= 10:            se, _ = sens_spec(g["prob"].values, g["y"].values); out.append(se)    return outconds = {    "baseline\n(light only)": fst6_sens("/kaggle/working/probs_seed[0-9].csv"),    "oversample\nFST 6":       fst6_sens("/kaggle/working/probs_seed*mitigate6*.csv"),    "train on\nFST 4,5":       fst6_sens("/kaggle/working/probs_seed*hold45*.csv"),}conds = {k: v for k, v in conds.items() if v}if len(conds) > 1:    fig, ax = plt.subplots(figsize=(4.0, 2.8))    ax.bar(list(conds), [np.mean(v) for v in conds.values()],           yerr=[np.std(v) for v in conds.values()], capsize=4,           color="#4C72B0", edgecolor="black", linewidth=.5)    ax.set_ylabel("FST 6 sensitivity"); ax.set_ylim(0, 1)    ax.set_title("Does more exposure fix FST 6?", fontsize=9)    plt.tight_layout(); plt.savefig("/kaggle/working/fig3_mitigation.png", bbox_inches="tight")    plt.show()    for k, v in conds.items():        print(f"{k.replace(chr(10),' '):28s} {np.mean(v):.3f} +/- {np.std(v):.3f}  (n={len(v)})")else:    print("mitigation/holdout runs not found yet -- figure 3 skipped")print("\nfigures saved to /kaggle/working/fig*.png")

## 8. What to take awayDownload from the output panel:- `probs_*.csv` — every per-image probability. All metrics recomputable without a GPU.- `results_*.json` — per-run summary metrics.- `fig1`–`fig3` — the paper's figures.**Caveats that belong in the paper, not hidden:**- Fitzpatrick17k labels come from dermatology atlases, not biopsy. Label noise  may itself correlate with skin tone — darker-skin cases in atlases are rarer  and more selectively photographed. Name this as an alternative explanation.- FST 1 is also low, so this is a *cliff at FST 6*, not a monotonic trend.  Do not draw a trend line through six points.- FST 6 has few positives. Report bootstrap CIs, not just seed variance.